# 04 - Prototipo Interactivo
**Rol:** Prototype Developer (PB-06)  
**Autor:** Miguel López  
**Dataset:** Employee Performance & Attrition (14,000 registros)  

Este notebook permite explorar el dataset de forma interactiva sin necesidad de escribir código. Está diseñado para que cualquier stakeholder pueda analizar las variables relacionadas con la rotación de empleados (attrition).

## 1. Importación de librerías
Cargamos las librerías necesarias: `pandas` para manipular datos, `matplotlib` y `seaborn` para visualizaciones, y `ipywidgets` para los controles interactivos.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
print('✅ Librerías cargadas correctamente')

## 2. Carga del dataset
Cargamos el dataset directamente desde la carpeta `data/raw/`. Se genera una copia con variables categóricas codificadas numéricamente para los gráficos que lo requieran.

In [ ]:
# Carga del dataset original
df2 = pd.read_csv('../data/raw/10-employee_performance.csv')

# Copia con variables categóricas codificadas
df = df2.copy()
for column in df.select_dtypes(include=['object']).columns:
    df[column] = LabelEncoder().fit_transform(df[column])

print(f'✅ Dataset cargado: {df2.shape[0]:,} filas × {df2.shape[1]} columnas')
print(f'🎯 Variable objetivo: attrition  |  0 = No rotó  |  1 = Sí rotó')
df2.head()

## 3. Prototipo interactivo

El panel está organizado en **5 pestañas**. Cada una contiene controles interactivos para explorar el dataset sin escribir código:

| Pestaña | Contenido | Widgets |
|---|---|---|
| Primeras Filas | Vista previa del dataset | — |
| Análisis Exploratorio | Estadísticas descriptivas y nulos | — |
| Scatter Plot | Dispersión entre dos variables | Dropdown |
| Violin | Distribución numérica por categoría | Dropdown + Slider |
| Histograma / Boxplot | Distribución con filtros | Dropdown + Slider + Checkbox + RadioButtons |

In [ ]:
# --- Outputs ---
output_head        = widgets.Output()
output_exploratory = widgets.Output()
output_scatter     = widgets.Output()
output_violin      = widgets.Output()
output_hist        = widgets.Output()

# --- Columnas ---
num_cols = [c for c in df2.select_dtypes(include='number').columns if c not in ['employee_id']]
cat_cols = df2.select_dtypes(include=['object', 'category']).columns.tolist()

# ── Widgets: Scatter ──────────────────────────────────────────
Dx = widgets.Dropdown(
    options=num_cols, value='monthly_salary',
    description='Variable X:', style={'description_width': 'initial'}
)
Dy = widgets.Dropdown(
    options=num_cols, value='age',
    description='Variable Y:', style={'description_width': 'initial'}
)

# ── Widgets: Violin ───────────────────────────────────────────
Dx2 = widgets.Dropdown(
    options=cat_cols, value='department',
    description='Categórica:', style={'description_width': 'initial'}
)
Dy2 = widgets.Dropdown(
    options=num_cols, value='monthly_salary',
    description='Numérica:', style={'description_width': 'initial'}
)
Op = widgets.FloatSlider(
    value=0.6, min=0.1, max=1.0, step=0.1,
    description='Opacidad:', continuous_update=False,
    readout_format='.1f', style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)

# ── Widgets: Histograma / Boxplot ─────────────────────────────
Dh = widgets.Dropdown(
    options=num_cols, value='monthly_salary',
    description='Variable:', style={'description_width': 'initial'}
)
Bins = widgets.IntSlider(
    value=30, min=5, max=80, step=5,
    description='Bins:', continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)
Chk = widgets.Checkbox(
    value=True, description='Separar por Attrition (0/1)', indent=False
)
Radio = widgets.RadioButtons(
    options=['Histograma', 'Boxplot'], value='Histograma',
    description='Tipo de gráfico:', style={'description_width': 'initial'}
)

# ── Funciones de graficado ────────────────────────────────────
def show_head():
    with output_head:
        output_head.clear_output()
        print(f'📋 Primeras 5 filas ({df2.shape[0]:,} filas × {df2.shape[1]} columnas):')
        display(df2.head())

def exploratory_analysis():
    with output_exploratory:
        output_exploratory.clear_output()
        print('📊 Descripción estadística del dataset:')
        display(df2.describe())
        print('\n🔍 Valores nulos por columna:')
        display(df2.isnull().sum().to_frame('Nulos'))

def create_scatter(change=None):
    x, y = Dx.value, Dy.value
    with output_scatter:
        output_scatter.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 5))
        colores = {0: '#4C72B0', 1: '#DD8452'}
        for val, label in [(0, 'No rotó'), (1, 'Sí rotó')]:
            subset = df2[df2['attrition'] == val]
            ax.scatter(subset[x], subset[y], label=label,
                       color=colores[val], alpha=0.4, s=15)
        ax.set_xlabel(x); ax.set_ylabel(y)
        ax.set_title(f'Dispersión: {x} vs {y}  (color = attrition)', fontweight='bold')
        ax.legend(title='Attrition')
        plt.tight_layout(); plt.show()

def create_violin(change=None):
    X, Y, alpha = Dx2.value, Dy2.value, Op.value
    with output_violin:
        output_violin.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.violinplot(data=df2, x=X, y=Y, ax=ax,
                       hue='attrition', split=False,
                       palette={0: '#4C72B0', 1: '#DD8452'},
                       alpha=alpha)
        ax.set_title(f'Violín: {Y} por {X}', fontweight='bold')
        ax.tick_params(axis='x', rotation=20)
        plt.tight_layout(); plt.show()

def create_hist(change=None):
    variable = Dh.value
    separar  = Chk.value
    bins     = Bins.value
    tipo     = Radio.value
    with output_hist:
        output_hist.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 5))
        hue_col = 'attrition' if separar else None
        palette = {0: '#4C72B0', 1: '#DD8452'}
        if tipo == 'Histograma':
            sns.histplot(data=df2, x=variable, hue=hue_col,
                         bins=bins, ax=ax, kde=True,
                         palette=palette if separar else None)
        else:
            sns.boxplot(data=df2, x=hue_col, y=variable, ax=ax,
                        palette=palette if separar else None)
        titulo = f'{tipo}: {variable}' + (' por Attrition' if separar else '')
        ax.set_title(titulo, fontweight='bold')
        plt.tight_layout(); plt.show()

# ── Conectar observers ────────────────────────────────────────
Dx.observe(create_scatter, names='value')
Dy.observe(create_scatter, names='value')
Dx2.observe(create_violin, names='value')
Dy2.observe(create_violin, names='value')
Op.observe(create_violin, names='value')
Dh.observe(create_hist, names='value')
Bins.observe(create_hist, names='value')
Chk.observe(create_hist, names='value')
Radio.observe(create_hist, names='value')

# ── Renderizar contenido inicial ──────────────────────────────
show_head()
exploratory_analysis()
create_scatter()
create_violin()
create_hist()

# ── Layout con Tabs ───────────────────────────────────────────
tab = widgets.Tab()
tab.children = [
    output_head,
    output_exploratory,
    widgets.VBox([
        widgets.HTML('<b>Selecciona dos variables numéricas (color = attrition)</b>'),
        widgets.HBox([Dx, Dy]),
        output_scatter
    ]),
    widgets.VBox([
        widgets.HTML('<b>Selecciona una variable categórica y una numérica</b>'),
        widgets.HBox([Dx2, Dy2]),
        Op,
        output_violin
    ]),
    widgets.VBox([
        widgets.HTML('<b>Selecciona variable, tipo de gráfico y filtros</b>'),
        widgets.HBox([Dh, Radio]),
        widgets.HBox([Bins, Chk]),
        output_hist
    ])
]

for i, t in enumerate(['Primeras Filas', 'Análisis Exploratorio',
                        'Scatter Plot', 'Violin', 'Histograma / Boxplot']):
    tab.set_title(i, t)

display(tab)

## 4. Conclusiones del prototipo

Este notebook interactivo permite explorar el dataset sin escribir código. Los principales patrones observables son:

- **Salario mensual:** los empleados que rotaron tienden a tener salarios menores
- **Horas extra:** mayor concentración de attrition en empleados con más horas extra
- **Satisfacción laboral:** valores bajos en `job_satisfaction` están asociados a mayor rotación
- **Balance vida-trabajo:** puntajes bajos en `work_life_balance_score` correlacionan con attrition

Estos patrones serán profundizados en los notebooks de modelado (Sprint 2).